In [2]:
from pathlib import Path

import pandas as pd
import sys

sys.path.append(str(Path.cwd().parent / "app"))

from vector_store import create_vector_store, search_similar_chunks
from document_loader import extract_text_from_pdf, chunk_document
from rag_pipeline import generate_answer

/home/jaydanng/Projects/rag-document-assistant/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load a test PDF

In [3]:
pdf_path = Path("../data/raw/sample/sample.pdf")

with open(pdf_path, "rb") as file:
    pages = extract_text_from_pdf(file)

chunks = chunk_document(pages, chunk_size=1000, chunk_overlap=200)

len(pages), len(chunks)

(30, 124)

## Create Vector Store

In [4]:
vector_store = create_vector_store(chunks)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6053.91it/s]


## Evaluation Questions

In [7]:
eval_questions = [
    {
        "question": "What is an LLM?",
        "expected_answer": "The document describes an LLM as a computational agent that can interact conversationally with people",
        "expected_page": 1,
    },
    {
        "question": "What are the 3 architectures for language models?",
        "expected_answer": "The document states the 3 architectures are the encoder, the decoder, and the encoder-decoder",
        "expected_page": 4,
    },
    {
        "question": "What is random sampling?",
        "expected_answer": "The document defines random sampling as randomly selecting a token to generate according to its probability in context as defined by the model",
        "expected_page": 11,
    },
    {
        "question": "What information is not provided in the document?",
        "expected_answer": "I don't know based on the provided document",
        "expected_page": None,
    },
    {
        "question": "What is temperature sampling?",
        "expected_answer": "Temperature sampling is the idea to reshape the probability distribution to increase the probability of high probability tokens and decrease the probability of the low probability tokens",
        "expected_page": 12,
    },
    {
        "question": "How do you train a large language model?",
        "expected_answer": "You train a large language model in 3 steps. Pretraining, where the model is trained to incrementally predict the next word in enourmous text corpora. Instruction tuning, where the model is trained to follow instructions, by giving it large amounts of text including instructions and answers to those instructions. Finally, alignment, where the model is trained to make it maximally helpful and less harmful",
        "expected_page": 14,
    },
    {
        "question": "What does finetuning mean?",
        "expected_answer": "Finetuning is the process of taking a fully pretrained model and running additional training passes on some new data",
        "expected_page": 18,
    },
    {
        "question": "How do you evaluate a large language model?",
        "expected_answer": "Large languages models are evaluated by using things like perplexity, where the lower perplexity of the model on the data, the better the model. You can also test it on downstream tasks such as question answering, machine translation, or reasoning.",
        "expected_page": 19,
    },
    {
        "question": "What are some ethical issues with large language models?",
        "expected_answer": "Some ethical concersn with large language models are that they can hallucinate which means they could say many things that are just false. The models could also suggest some unsafe actions or by verbally attacking users. Another issue is privacy or emotional dependance on a model",
        "expected_page": 22,
    },
]

## Run Evaluation

In [8]:
results = []

for item in eval_questions:
    question = item["question"]
    expected_page = item["expected_page"]

    retrieved = search_similar_chunks(vector_store, question, k=4)
    retrieved_docs = [doc for doc, score in retrieved]

    answer = generate_answer(question, retrieved_docs)

    retrieved_pages = [
        doc.metadata.get("page_number")
        for doc, score in retrieved
    ]

    correct_source_retrieved = (
        expected_page in retrieved_pages
        if expected_page is not None
        else None
    )

    results.append({
        "question": question,
        "expected_answer": item["expected_answer"],
        "generated_answer": answer,
        "expected_page": expected_page,
        "retrieved_pages": retrieved_pages,
        "correct_source_retrieved": correct_source_retrieved,
    })

eval_df = pd.DataFrame(results)
eval_df

,question,expected_answer,generated_answer,expected_page,retrieved_pages,correct_source_retrieved
0,What is an LLM?,The document describes an LLM as a computation...,"Based on the context provided, an LLM (Large L...",1.0,"[24, 1, 1, 5]",True
1,What are the 3 architectures for language models?,The document states the 3 architectures are th...,"According to the context, there are three comm...",4.0,"[4, 24, 27, 2]",True
2,What is random sampling?,The document defines random sampling as random...,"Random sampling, also known as random multinom...",11.0,"[12, 12, 11, 11]",True
3,What information is not provided in the document?,I don't know based on the provided document,I don't know based on the provided document.,NaN,"[17, 17, 23, 18]",None
4,What is temperature sampling?,Temperature sampling is the idea to reshape th...,Temperature sampling is a method that reshapes...,12.0,"[12, 12, 13, 25]",True
5,How do you train a large language model?,You train a large language model in 3 steps. P...,"Based on the provided document, it appears tha...",14.0,"[2, 4, 3, 19]",False
6,What does finetuning mean?,Finetuning is the process of taking a fully pr...,"According to Source 2 | Page 19, fine-tuning r...",18.0,"[29, 19, 17, 7]",False
7,How do you evaluate a large language model?,Large languages models are evaluated by using ...,"According to the provided context, there are m...",19.0,"[19, 21, 22, 25]",True
8,What are some ethical issues with large langua...,Some ethical concersn with large language mode...,"According to the context, some ethical issues ...",22.0,"[22, 24, 18, 2]",True


In [9]:
valid_retrievals = eval_df["correct_source_retrieved"].dropna()

retrieval_accuracy = valid_retrievals.mean()

print(f"Retrieval accuracy: {retrieval_accuracy:.2%}")

Retrieval accuracy: 75.00%


## Manual Answer Grading

In [27]:
eval_df

,question,expected_answer,generated_answer,expected_page,retrieved_pages,correct_source_retrieved,answer_grounded,answer_correct
0,What is an LLM?,The document describes an LLM as a computation...,"Based on the context provided, an LLM (Large L...",1.0,"[24, 1, 1, 5]",True,Yes,Yes
1,What are the 3 architectures for language models?,The document states the 3 architectures are th...,"According to the context, there are three comm...",4.0,"[4, 24, 27, 2]",True,Yes,Yes
2,What is random sampling?,The document defines random sampling as random...,"Random sampling, also known as random multinom...",11.0,"[12, 12, 11, 11]",True,Yes,Yes
3,What information is not provided in the document?,I don't know based on the provided document,I don't know based on the provided document.,NaN,"[17, 17, 23, 18]",None,Yes,Yes
4,What is temperature sampling?,Temperature sampling is the idea to reshape th...,Temperature sampling is a method that reshapes...,12.0,"[12, 12, 13, 25]",True,Yes,Yes
5,How do you train a large language model?,You train a large language model in 3 steps. P...,"Based on the provided document, it appears tha...",14.0,"[2, 4, 3, 19]",False,Yes,Partial
6,What does finetuning mean?,Finetuning is the process of taking a fully pr...,"According to Source 2 | Page 19, fine-tuning r...",18.0,"[29, 19, 17, 7]",False,Yes,Partial
7,How do you evaluate a large language model?,Large languages models are evaluated by using ...,"According to the provided context, there are m...",19.0,"[19, 21, 22, 25]",True,Yes,Yes
8,What are some ethical issues with large langua...,Some ethical concersn with large language mode...,"According to the context, some ethical issues ...",22.0,"[22, 24, 18, 2]",True,Yes,Yes
